# Tutorial 7 · Intrinsically interpretable models

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Interpretable machine learning and baselines

You will fit and read four models whose structure is their explanation: an **L2-logistic regression**, a **depth-limited decision tree**, a **k-nearest-neighbours classifier** and an **explainable boosting machine (EBM)**. Each is scored on **people it has not seen**, beside a **zero-rate baseline**, with **95% bootstrap intervals over people**. A random forest at the end is the black box they are compared with: how much accuracy does reading the model cost here?

**With the course coach.** Run *T7.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T7.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T7.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


## 0) Setup & data


In [ ]:
# T7.01 Install interpret
%pip install -q interpret


In [ ]:
# T7.02 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T7.03 Load and split by person
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, log_loss

RANDOM_STATE = 42

data_path = Path('/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/')   # Jupyter on your own computer: the folder that holds the file
df = pd.read_csv(data_path / 'features_df_10s_3s.csv')
target_col = 'activity'

# Features are measurements only: who the person is, which trial and which window are not.
meta_cols = ['activity', 'subject', 'trial', 'window']
feature_cols = [c for c in df.columns if c not in meta_cols]
X = df[feature_cols].copy()
y = df[target_col].copy()
subjects = df['subject'].to_numpy()

# Split by person: every window of a person is on one side.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=subjects))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
subjects_train, subjects_test = subjects[train_idx], subjects[test_idx]
print('Train:', X_train.shape, f'from {len(np.unique(subjects_train))} people · Test:', X_test.shape,
      f'from {len(np.unique(subjects_test))} people · on both sides:', len(set(subjects_train) & set(subjects_test)))


## Baseline first

Before any model: what does always predicting the commonest training activity score on the test people? Every number below is read against it.


In [ ]:
# T7.04 Zero-rate baseline
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)
zero_rate = {'accuracy': accuracy_score(y_test, pred_dummy), 'balanced_accuracy': balanced_accuracy_score(y_test, pred_dummy),
             'f1_macro': f1_score(y_test, pred_dummy, average='macro')}
print(f"Always '{dummy.classes_[dummy.class_prior_.argmax()]}':", {k: round(v, 3) for k, v in zero_rate.items()})


In [ ]:
# T7.05 Report the split and the baseline
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T7.00).
if coach:
    coach.baseline(y_train)                     # records that a baseline exists
    coach.report((train_idx, test_idx), subjects, y.to_numpy(), tutorial="T7",
                 lesson={"zero_rate_score": zero_rate['accuracy']})
else:
    print("Not connected: nothing sent. Run T7.00 with your code to use the coach.")


### Helper: reliability diagram & ECE


In [ ]:
# T7.06 Helper: reliability diagram and ECE
def reliability_ece(y_true, proba, n_bins=15, classes=None, y_pred=None, binning='equal_width'):
    """
    Top-label reliability diagram + ECE.
    - y_true: array of true labels (strings or ints)
    - proba:  (n_samples, n_classes)
    - classes: optional array of class labels matching proba columns (e.g., model.classes_)
    - y_pred: optional array of predicted labels (same dtype as y_true). If provided, we use it.
    - binning: 'equal_width' or 'equal_freq'
    """
    import numpy as np
    import matplotlib.pyplot as plt

    y_true = np.asarray(y_true)
    conf = proba.max(axis=1)

    if y_pred is None:
        idx = proba.argmax(axis=1)
        if classes is not None:
            y_pred = np.asarray(classes)[idx]     # map to label dtype
        else:
            y_pred = idx                          # only safe if y_true are ints 0..K-1
    else:
        y_pred = np.asarray(y_pred)

    correct = (y_pred == y_true).astype(int)

    # binning
    if binning == 'equal_freq':
        qs = np.linspace(0, 1, n_bins+1)
        edges = np.quantile(conf, qs)
        edges[0], edges[-1] = 0.0, 1.0
        idx_bin = np.searchsorted(edges, conf, side='right') - 1
    else:
        edges = np.linspace(0, 1, n_bins+1)
        idx_bin = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)   # confidence 1.0 in the last bin

    ece, xs, ys = 0.0, [], []
    for b in range(n_bins):
        m = (idx_bin == b)
        if m.sum() == 0:
            continue
        acc_b = correct[m].mean()
        conf_b = conf[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc_b - conf_b)
        xs.append(conf_b); ys.append(acc_b)

    plt.figure()
    plt.plot([0, 1], [0, 1], '--')
    plt.plot(xs, ys, marker='o')
    plt.xlabel('Mean confidence (per bin)')
    plt.ylabel('Empirical accuracy (per bin)')
    plt.title(f'Reliability diagram (ECE={ece:.4f})')
    plt.show()
    return float(ece)


### Helper: metric evaluation & bootstrap confidence intervals


In [ ]:
# T7.07 Helper: metrics and bootstrap intervals
from sklearn.metrics import accuracy_score, f1_score, log_loss

def _bin_indices(conf, n_bins=50, binning='equal_freq'):
    """Return per-sample bin index for confidence values."""
    conf = np.asarray(conf, dtype=float)
    if binning == 'equal_freq':
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(conf, qs)
        edges[0], edges[-1] = 0.0, 1.0
        # If many duplicates (e.g., confidences all the same), fall back to equal-width
        if np.unique(edges).size < edges.size:
            edges = np.linspace(0, 1, n_bins + 1)
        idx = np.searchsorted(edges, conf, side='right') - 1
    else:  # 'equal_width'
        edges = np.linspace(0, 1, n_bins + 1)
        idx = np.digitize(conf, edges) - 1
    # keep in range [0, n_bins-1]
    idx = np.clip(idx, 0, n_bins - 1)
    return idx, n_bins

def ece_only(y_true, proba, n_bins=50, *, classes=None, y_pred=None, binning='equal_freq'):
    """
    Top-label ECE with robust label handling and flexible binning.
    - y_true: array of true labels (strings or ints)
    - proba : (n_samples, n_classes) predicted probabilities
    - classes: optional sequence matching proba columns (e.g., model.classes_)
    - y_pred: optional predicted labels (same dtype as y_true). If not given,
              we map argmax indices to labels via `classes` when provided.
    - binning: 'equal_freq' (default) or 'equal_width'
    """
    y_true = np.asarray(y_true)
    conf = proba.max(axis=1)

    if y_pred is None:
        top_idx = proba.argmax(axis=1)
        y_pred = (np.asarray(classes)[top_idx] if classes is not None else top_idx)
    else:
        y_pred = np.asarray(y_pred)

    correct = (y_pred == y_true).astype(int)
    idx, B = _bin_indices(conf, n_bins=n_bins, binning=binning)

    ece = 0.0
    for b in range(B):
        m = (idx == b)
        if m.sum() == 0:
            continue
        acc_b  = correct[m].mean()
        conf_b = conf[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc_b - conf_b)
    return float(ece)

def classwise_ece(y_true, proba, classes, n_bins=50, binning='equal_freq'):
    """
    Compute classwise ECE (OvR) and macro average.
    y_true : array-like of true labels (strings or ints)
    proba  : (n_samples, n_classes) predicted probabilities
    classes: sequence of class labels matching proba columns (e.g., model.classes_)
    """
    y_true = np.asarray(y_true)
    classes = np.asarray(classes)
    K = proba.shape[1]
    rows = []
    eces = []

    for k in range(K):
        p_k = proba[:, k]
        pos = (y_true == classes[k]).astype(int)   # one-vs-rest ground truth
        idx, B = _bin_indices(p_k, n_bins=n_bins, binning=binning)

        ece_k = 0.0
        for b in range(B):
            m = (idx == b)
            if m.sum() == 0:
                continue
            acc_b  = pos[m].mean()
            conf_b = p_k[m].mean()
            ece_k += (m.sum() / len(y_true)) * abs(acc_b - conf_b)

        eces.append(ece_k)
        rows.append({
            'class': classes[k],
            'ece': float(ece_k),
            'prevalence': float(pos.mean()),
        })

    macro_ece = float(np.mean(eces))
    table = pd.DataFrame(rows).sort_values('ece', ascending=False).reset_index(drop=True)
    return macro_ece, table

def plot_ece_bar(ece_table, title='Classwise ECE'):
    plt.figure()
    labels = [str(c) for c in ece_table['class']]
    plt.bar(labels, ece_table['ece'].values)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('ECE')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_classwise_reliability(y_true, proba, classes, class_subset=None, n_bins=10, binning='equal_freq'):
    """
    Plot one reliability diagram per class (OvR). By default shows all classes;
    pass class_subset (list of labels) to limit to e.g. the worst 6 by ECE.
    """
    y_true = np.asarray(y_true)
    classes = np.asarray(classes)
    if class_subset is None:
        class_subset = classes

    for c in class_subset:
        k = list(classes).index(c)
        p_k = proba[:, k]
        pos = (y_true == classes[k]).astype(int)

        idx, B = _bin_indices(p_k, n_bins=n_bins, binning=binning)
        xs, ys = [], []
        ece = 0.0
        for b in range(B):
            m = (idx == b)
            if m.sum() == 0:
                continue
            acc_b  = pos[m].mean()
            conf_b = p_k[m].mean()
            ece += (m.sum()/len(y_true)) * abs(acc_b - conf_b)
            xs.append(conf_b); ys.append(acc_b)

        plt.figure()
        plt.plot([0,1],[0,1],'--')
        plt.plot(xs, ys, marker='o')
        plt.xlabel('Mean confidence for class "{}"'.format(c))
        plt.ylabel('Empirical accuracy (is_y==class)')
        plt.title('Classwise reliability — {} (ECE={:.4f})'.format(c, ece))
        plt.show()

def compute_metrics(y_true, proba, preds, *, average='binary', labels=None, binning='equal_freq'):
    """
    Metrics with safe NLL label alignment and robust ECE.
    - labels: pass model.classes_ in multiclass to align log_loss columns
    - binning: forwarded to ece_only
    """
    y_true = np.asarray(y_true)
    preds  = np.asarray(preds)
    acc = accuracy_score(y_true, preds)
    f1  = f1_score(y_true, preds, average=average)
    try:
        nll = log_loss(y_true, proba, labels=labels)
    except Exception:
        nll = np.nan
    # Use preds directly to avoid dtype mismatches inside ECE
    ece = ece_only(y_true, proba, n_bins=50, classes=labels, y_pred=preds, binning=binning)
    return {'accuracy': acc, 'f1': f1, 'nll': float(nll), 'ece': ece}

def bootstrap_ci(y_true, proba, preds, *, groups=None, average='binary', labels=None, B=300, seed=42, binning='equal_freq'):
    """
    Bootstrap CIs for Accuracy, F1, NLL, ECE.
    - groups: the person of each row. Given, whole people are resampled (Tutorial 5); without it,
              rows are, and the interval says nothing about new people.
    - labels: pass model.classes_ when available (multiclass)
    - binning: forwarded to ece_only
    """
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); n = len(y_true)
    if groups is not None:
        groups = np.asarray(groups); people = np.unique(groups)
        rows = {p: np.flatnonzero(groups == p) for p in people}
    accs, f1s, nlls, eces = [], [], [], []
    for _ in range(B):
        if groups is None:
            idx = rng.randint(0, n, size=n)
        else:
            idx = np.concatenate([rows[p] for p in rng.choice(people, size=len(people), replace=True)])
        m = compute_metrics(y_true[idx], proba[idx], preds[idx],
                            average=average, labels=labels, binning=binning)
        accs.append(m['accuracy']); f1s.append(m['f1'])
        nlls.append(m['nll']);      eces.append(m['ece'])
    df = pd.DataFrame({'accuracy': accs, 'f1': f1s, 'nll': nlls, 'ece': eces})
    return df.describe(percentiles=[0.025, 0.5, 0.975]).T[['mean', 'std', '2.5%', '50%', '97.5%']]


## A) L2-logistic regression · shrinkage and calibration

An L2 penalty shrinks every coefficient towards zero but sets none of them to zero; an L1 penalty does (the exercise). Features are standardised on the training people first, so coefficients are comparable: one unit is one standard deviation.


In [ ]:
# T7.08 L2-logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train)
Xte_s = scaler.transform(X_test)

logreg = LogisticRegression(C=1.0, max_iter=2000, random_state=42)   # L2, multinomial for several classes
logreg.fit(Xtr_s, y_train)

proba_lr = logreg.predict_proba(Xte_s)
pred_lr  = logreg.predict(Xte_s)

# Metrics (robust to helper signature)
avg = 'binary' if len(np.unique(y)) <= 2 else 'macro'
try:
    # Preferred: your helper supports labels=
    m_lr = compute_metrics(y_test.to_numpy(), proba_lr, pred_lr,
                           average=avg, labels=logreg.classes_)
except TypeError:
    # Fallback: helper without labels=; patch NLL afterward
    m_lr = compute_metrics(y_test.to_numpy(), proba_lr, pred_lr, average=avg)
    try:
        m_lr['nll'] = float(log_loss(y_test, proba_lr, labels=logreg.classes_))
    except Exception:
        pass

print("Multinomial Logistic (L2, lbfgs) metrics:", m_lr)
print("Coefficients exactly zero:", int((logreg.coef_ == 0).sum()), "of", logreg.coef_.size)
unscaled = int((np.abs(Xtr_s.std(axis=0) - 1) > 0.1).sum())   # features the model saw on their raw scale
print("Features not standardised:", unscaled)

# Reliability / ECE (top-label)
ece_lr = reliability_ece(
    y_test.to_numpy(),
    proba_lr,
    classes=logreg.classes_,   # maps argmax to labels if y_pred not given
    y_pred=pred_lr,            # safest: pass the labels directly
    n_bins=50,
    binning='equal_freq'       # nicer, more stable binning
)


In [ ]:
# T7.09 Read the coefficients
n_classes = len(logreg.classes_)

# Make sure feature names match the fitted dimension
feature_names = list(feature_cols)
n_feat = logreg.coef_.shape[1]
if len(feature_names) != n_feat:
    feature_names = [f"f{i}" for i in range(n_feat)]

if n_classes == 2:
    # Binary logistic: standard odds interpretation is valid
    beta = pd.Series(logreg.coef_.ravel(), index=feature_names)
    odds = np.exp(beta)  # multiplicative change in odds of class=classes_[1] per +1 (standardized) unit

    tbl = (pd.DataFrame({
            'coef': beta,
            'odds_ratio (class={} vs {})'.format(logreg.classes_[1], logreg.classes_[0]): odds
        })
        .sort_values('coef', key=lambda s: s.abs(), ascending=False))

    display(tbl.head(15))

    plt.figure()
    plt.bar(tbl.head(15).index.astype(str), tbl.head(15)['coef'].values)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Coefficient (β)  — binary logistic')
    plt.title('Top coefficients (by |β|) — Logistic (binary, L2)')
    plt.tight_layout()
    plt.show()

else:
    # Multinomial (softmax) logistic: coef_ is K x p.
    # A safe global importance is the L2 norm across classes for each feature.
    coef_mat = pd.DataFrame(logreg.coef_, index=logreg.classes_, columns=feature_names)

    global_imp = pd.Series(
        np.linalg.norm(coef_mat.values, axis=0), index=feature_names
    ).sort_values(ascending=False)

    print("Global feature importance (L2 norm across class-wise coefficients):")
    display(global_imp.head(15).to_frame('L2_norm'))

    # Per-class top features
    top_k = 10
    print(f"\nTop {top_k} features per class (by |β|):")
    for cls in coef_mat.index:
        s = coef_mat.loc[cls].abs().sort_values(ascending=False).head(top_k)
        out = pd.DataFrame({
            'coef': coef_mat.loc[cls, s.index],
            'abs_coef': s
        }).assign(class_=cls)
        display(out)

    # Bar plot of global top features
    top = global_imp.head(15)
    plt.figure()
    plt.bar(top.index.astype(str), top.values)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Global importance (L2 norm across classes)')
    plt.title('Multinomial Logistic — Top features (global)')
    plt.tight_layout()
    plt.show()

    # Optional: pairwise odds-style view relative to a baseline class.
    # For multinomial, log-odds between class c and baseline b for feature j is (β_cj - β_bj).
    # Exponentiating gives multiplicative change in odds (c vs b) per +1 standardized unit.
    baseline = logreg.classes_[0]
    rel_odds = np.exp(coef_mat.sub(coef_mat.loc[baseline], axis=1))  # shape K x p
    print(f"\nPairwise odds multipliers vs baseline class = '{baseline}' (exp(β_c - β_baseline)) "
          f"for top 5 global features:")
    display(rel_odds[top.index[:5]].round(3))


In [ ]:
# T7.10 Logistic: intervals over people
# 95% bootstrap CIs
ci_lr = bootstrap_ci(y_test.to_numpy(), proba_lr, pred_lr, groups=subjects_test,
                     average=avg, labels=logreg.classes_, B=300, seed=42, binning='equal_freq')
display(ci_lr)


## B) Decision Tree


In [ ]:
# T7.11 Decision tree
tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

proba_tr = tree.predict_proba(X_test)
pred_tr  = tree.predict(X_test)

avg = 'binary' if len(np.unique(y))<=2 else 'macro'
m_tr = compute_metrics(y_test.to_numpy(), proba_tr, pred_tr, average=avg)
print("Decision Tree metrics:", m_tr)

# Readable text rules (no Graphviz dependency)
text_rules = export_text(tree, feature_names=list(feature_cols))
print(text_rules)


**Choosing the depth.** Depth is the dial between a tree you can read and one that fits. Choose it with the **training people** only, by cross-validation that keeps each person in one fold; scoring every depth on the test people would turn them into a validation set.


In [ ]:
# T7.12 Depth chosen on training people
depths = list(range(2, 10))
cv = GroupKFold(n_splits=5)
cv_scores = []
for d in depths:
    fold = []
    for tr, va in cv.split(X_train, y_train, groups=subjects_train):
        t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=RANDOM_STATE).fit(X_train.iloc[tr], y_train.iloc[tr])
        fold.append(balanced_accuracy_score(y_train.iloc[va], t.predict(X_train.iloc[va])))
    cv_scores.append(np.mean(fold))
best_depth = depths[int(np.argmax(cv_scores))]

plt.figure()
plt.plot(depths, cv_scores, marker='o')
plt.xlabel('Max depth'); plt.ylabel('Balanced accuracy (5 folds of training people)')
plt.title(f'Decision tree: depth chosen on training people ({best_depth})'); plt.show()

# 95% bootstrap intervals over people, for the depth-5 tree above
ci_tr = bootstrap_ci(y_test.to_numpy(), proba_tr, pred_tr, groups=subjects_test, average=avg, B=300, seed=RANDOM_STATE)
print("Decision Tree — 95% bootstrap intervals over people"); display(ci_tr)


## C) KNN Classifier


In [ ]:
# T7.13 k-nearest neighbours
# === KNN classifier (instance-based, distance-weighted) ===
from sklearn.neighbors import KNeighborsClassifier

# Choose k (you can tweak). Distance weighting usually gives better probabilities.
K = 20
knn = KNeighborsClassifier(n_neighbors=K, weights='distance', n_jobs=-1)
knn.fit(Xtr_s, y_train)

proba_knn = knn.predict_proba(Xte_s)
pred_knn  = knn.predict(Xte_s)

avg = 'binary' if len(np.unique(y)) <= 2 else 'macro'

# Metrics (+ aligned labels for NLL)
m_knn = compute_metrics(
    y_test.to_numpy(), proba_knn, pred_knn,
    average=avg, labels=knn.classes_, binning='equal_freq'
)
print(f"KNN (k={K}, distance-weighted) metrics:", m_knn)

# Reliability diagram / ECE
_ = reliability_ece(
    y_test.to_numpy(), proba_knn,
    classes=knn.classes_, y_pred=pred_knn,
    n_bins=50, binning='equal_freq'
)

# 95% bootstrap CIs
ci_knn = bootstrap_ci(
    y_test.to_numpy(), proba_knn, pred_knn, groups=subjects_test,
    average=avg, labels=knn.classes_, B=300, seed=42, binning='equal_freq'
)
print(f"KNN (k={K}) — 95% bootstrap CIs"); display(ci_knn)

# Classwise ECE (which classes are miscalibrated?)
macro_ece_knn, ece_table_knn = classwise_ece(
    y_test.to_numpy(), proba_knn, knn.classes_,
    n_bins=50, binning='equal_freq'
)
# print("KNN classwise macro-ECE:", macro_ece_knn)
display(ece_table_knn)
plot_ece_bar(ece_table_knn, title=f'KNN (k={K}) — Classwise ECE')


In [ ]:
# T7.14 Whose windows are the neighbours?
# Inspect the top neighbors for interpretability (choose any test index i)
i = 0
dist, idx = knn.kneighbors(Xte_s[i:i+1], n_neighbors=K, return_distance=True)
nbr_idx = idx[0]
nbr = pd.DataFrame({
    'train_idx': nbr_idx,
    'distance': dist[0],
    'neighbor_label': y_train.to_numpy()[nbr_idx],
    'neighbor_person': subjects_train[nbr_idx],
}).sort_values('distance')
print("Test sample true / predicted:", y_test.iloc[i], pred_knn[i])
display(nbr.head(20))
print('Test person:', subjects_test[i], '· neighbours come from', nbr['neighbor_person'].nunique(), 'training people')


## D) EBM — Explainable Boosting Machine


In [ ]:
# T7.15 Fit the EBM
from interpret.glassbox import ExplainableBoostingClassifier

ebm = ExplainableBoostingClassifier(interactions=0, n_jobs=-1, random_state=RANDOM_STATE)
ebm.fit(X_train, y_train)


In [ ]:
# T7.16 EBM metrics
proba_ebm = ebm.predict_proba(X_test)
pred_ebm  = ebm.predict(X_test)

avg = 'binary' if len(np.unique(y))<=2 else 'macro'
m_ebm = compute_metrics(y_test.to_numpy(), proba_ebm, pred_ebm, average=avg)
print("EBM metrics:", m_ebm)
    
# 95% bootstrap CIs
ci_ebm = bootstrap_ci(y_test.to_numpy(), proba_ebm, pred_ebm, groups=subjects_test, average=avg, B=300, seed=RANDOM_STATE)
print("EBM — 95% bootstrap CIs"); display(ci_ebm)


In [ ]:
# T7.17 EBM: global shape functions
from interpret import show
show(ebm.explain_global())


In [ ]:
# T7.18 EBM: one window explained
show(ebm.explain_local(X_test.iloc[:10], y_test.iloc[:10]), 0)


## Model comparison · what does reading cost?

All rows: the same test people. The random forest is the black box; the gap between it and the best glass box is what interpretability costs here, to be read against the intervals above.


In [ ]:
# T7.19 Compare with a black box
rows = []
avg = 'binary' if len(np.unique(y))<=2 else 'macro'
rows.append({'model':'L2-Logistic', **compute_metrics(y_test.to_numpy(), proba_lr, pred_lr, average=avg)})
rows.append({'model':'Decision Tree', **compute_metrics(y_test.to_numpy(), proba_tr, pred_tr, average=avg)})
rows.append({'model':'KNN Classifier', **compute_metrics(y_test.to_numpy(), proba_knn, pred_knn, average=avg)})
rows.append({'model':'EBM', **compute_metrics(y_test.to_numpy(), proba_ebm, pred_ebm, average=avg)})

from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
proba_rf, pred_rf = forest.predict_proba(X_test), forest.predict(X_test)
rows.append({'model':'Random forest (black box)', **compute_metrics(y_test.to_numpy(), proba_rf, pred_rf, average=avg)})
rows.insert(0, {'model': 'Zero-rate', 'accuracy': zero_rate['accuracy'], 'f1': zero_rate['f1_macro']})

summary_df = pd.DataFrame(rows)
bal = {name: balanced_accuracy_score(y_test, p) for name, p in
       [('logistic', pred_lr), ('tree', pred_tr), ('knn', pred_knn), ('ebm', pred_ebm), ('forest', pred_rf)]}
best_glass = max(('logistic', 'tree', 'knn', 'ebm'), key=bal.get)
glass_gap = bal['forest'] - bal[best_glass]
display(summary_df)
print(f'Balanced accuracy: forest {bal["forest"]:.3f}, best glass box ({best_glass}) {bal[best_glass]:.3f}, gap {glass_gap:+.3f}')


In [ ]:
# T7.20 Report the comparison
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T7.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y.to_numpy(), model=logreg, tutorial="T7",
                 scores={f"balanced_accuracy_{k}": v for k, v in bal.items()},
                 lesson={"glass_box_gap": glass_gap, "unscaled_feature_count": unscaled})
else:
    print("Not connected: nothing sent. Run T7.00 with your code to use the coach.")


## ✍️ Exercise 

**Goal:** Meet an **interpretability budget** with one intrinsic model and justify the trade-off.

1) Pick a budget:  
   - Logistic: ≤ **n** nonzero coefficients (choose coefficient C [0.1, 1, 10] for L1 penalization; L1 needs `solver='saga'`).  
   - Decision Tree: depth **3** ≤ d ≤ **7** and min_samples_leaf **3** ≤ m ≤ **7**. 
   - KNN Classifier: choose **k** from a [5, 10, 20].
   - EBM: ≤ **3** max_leaves and **[0, 0.5x, 1x]** interaction.

2) Tune hyperparameters with cross-validation **by person** (`GroupKFold` on `subjects_train`) to satisfy the budget. Report test **Accuracy**, **F1**, **NLL**, **ECE** and their **95% bootstrap intervals over people** (`bootstrap_ci(..., groups=subjects_test)`), beside the zero-rate baseline.

3) Score the test people **once**, for the model you chose. Is the gap to the forest larger than the intervals?
